## **ECE 226: Optimize&Accelerate/Platforms, Fall 2026, UCSD**
## **Lab 0: Meet Your Accelerator**
### **Instructor: Prof. Farinaz Koushanfar**
### **Due: Unlimited — no credit**

---


## Goals

Lab 0 is not graded on the absolute numbers you get — it is graded on whether you
produced **your own device's** roofline and can read it. Everything in Labs 1 through 4
refers back to the two numbers you measure here.

- Query whatever accelerator you have, from Python
- **Measure** your compute ceiling, then read Lecture 4's peak-FLOP/s formula *backwards*
  to see what that ceiling implies per compute unit per clock
- **Measure** your achievable memory bandwidth, and explain the shape of the curve
- Sweep GEMM shapes and precisions, and place each measurement on the roofline you built
- Profile one training step of the ResNet you already know, and find where the time went

## Instructions

- All solutions must be written in this notebook, in Python.
- Submit the notebook as `.ipynb` **and** as a PDF with outputs inline.
- **Any machine will do.** The class container on the shared GPU is the fastest path, but
  this notebook is written to also run on a gaming laptop, on an M-series Mac, or on a
  laptop with no GPU at all. Every sweep sizes itself to the machine it finds, so no cell
  should take more than a couple of minutes even on a CPU.
- Absolute numbers will differ by three orders of magnitude between a classmate's laptop
  and MI300X. **The shape of every result should not.** 
- A few cells can only report GPU-specific facts. Without a GPU they print
  *"this cell would only be useful if connected to a GPU"* and skip. That is expected
  output.
- For now I suggest you use DSMLP cloud for this project since it provides you with an nvidia gpu. The notebook has been tested only for that specific infrastructure.

## Setup


In [2]:
import os, re, math, time, json, platform, shutil, subprocess
import numpy as np
import torch
import torch.nn as nn
from torch.optim import SGD
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0)


def pick_device():
    """First accelerator we can actually use; CPU if there is none.

    Set LAB0_DEVICE=cpu to force the host, which is how you compare your GPU
    against your own CPU -- run the notebook twice and keep both tables.
    """
    want = os.environ.get("LAB0_DEVICE", "").strip().lower()
    if want:
        kind = "rocm" if (want == "cuda" and torch.version.hip) else want
        return torch.device(want), kind
    if torch.cuda.is_available():
        return torch.device("cuda"), ("rocm" if torch.version.hip else "cuda")
    mps = getattr(torch.backends, "mps", None)
    if mps is not None and mps.is_available():
        return torch.device("mps"), "mps"
    for name in ("xpu", "npu"):            # Intel GPUs, vendor out-of-tree backends
        be = getattr(torch, name, None)
        if be is not None and getattr(be, "is_available", lambda: False)():
            return torch.device(name), name
    return torch.device("cpu"), "cpu"


DEV, ACCEL = pick_device()
HAS_GPU = ACCEL != "cpu"
IS_CUDA = DEV.type == "cuda"        # True on ROCm too: PyTorch shares the namespace


def sync():
    """Accelerators are asynchronous. Never time one without this."""
    if IS_CUDA:
        torch.cuda.synchronize()
    elif DEV.type == "mps":
        fn = getattr(getattr(torch, "mps", None), "synchronize", None)
        if fn is not None:
            fn()


NO_GPU_MSG = (
    "This cell would only be useful if connected to a GPU.\n"
    "No GPU is visible, so there is nothing for it to report --\n"
    "please proceed with the rest of the exercise."
)


def gpu_only():
    """True when a GPU-only cell should skip; prints the standard notice."""
    if HAS_GPU:
        return False
    print(NO_GPU_MSG)
    return True


def host_name():
    """A readable name for the CPU, for labelling plots on a laptop run."""
    try:
        if platform.system() == "Darwin":
            return subprocess.run(["sysctl", "-n", "machdep.cpu.brand_string"],
                                  capture_output=True, text=True, timeout=10).stdout.strip()
        if platform.system() == "Linux":
            for line in open("/proc/cpuinfo"):
                if line.startswith("model name"):
                    return line.split(":", 1)[1].strip()
    except Exception:
        pass
    return platform.processor() or platform.machine() or "unknown CPU"


print("torch             :", torch.__version__)
print("torch.version.hip :", torch.version.hip)      # a string on ROCm, None on CUDA
print("torch.version.cuda:", torch.version.cuda)     # None on ROCm
print("backend chosen    :", ACCEL, "->", DEV)
print("host              :", platform.platform(), "/", platform.machine())
if not HAS_GPU:
    print("\nNo GPU visible. Every cell below still runs -- on the CPU, at CPU speed.")
    print("The sizes auto-scale, so nothing should take minutes. Cells that can only")
    print("report GPU-specific facts will say so and skip.")


torch             : 2.8.0+cu126
torch.version.hip : None
torch.version.cuda: 12.6
backend chosen    : cuda -> cuda
host              : Linux-6.8.0-138-generic-x86_64-with-glibc2.35 / x86_64


### Sizing the benchmarks to the machine (provided)

Read this cell, do not rewrite it. Every sweep in this notebook picks its own sizes by
*measuring* how fast this machine is, so the same notebook finishes in a minute on a
laptop and still saturates a 304-CU datacenter GPU.

Two things worth stealing for your own work later:

- **Grow until you run out of time or memory.** A GEMM's cost grows as $N^3$; doubling
  $N$ eight times covers four orders of magnitude of work.
- **The crude timer below is not a benchmark.** It fires once, with no warm-up and no
  median. It is good enough to choose sizes and nothing else — Q2 asks you to write the
  real harness.
- `candidate_dtypes()` is here rather than in your code for one reason: on Apple Silicon
  with PyTorch older than 2.3, merely *creating* a bfloat16 tensor raises a C++ error that
  kills the kernel instead of raising a Python exception. You cannot discover that with
  `try`/`except`, so it is filtered out in advance. Knowing which failures your language
  can catch is part of the job.


In [3]:
# One knob. Raise it for tighter numbers, lower it if a cell takes too long.
BUDGET_MS = float(os.environ.get("LAB0_BUDGET_MS", "150"))   # per timed call


def free_bytes():
    """Roughly how much device memory we may help ourselves to."""
    if IS_CUDA:
        try:
            return torch.cuda.mem_get_info()[0]
        except Exception:
            return torch.cuda.get_device_properties(0).total_memory // 2
    rec = getattr(getattr(torch, "mps", None), "recommended_max_memory", None)
    if rec is not None:
        try:
            return int(rec())
        except Exception:
            pass
    try:                        # host RAM, quarter of it, for a CPU run
        return int(0.25 * os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES"))
    except Exception:
        return 2 * 2**30        # conservative: somebody in the class has an 8 GB laptop


def free_cache():
    if IS_CUDA:
        torch.cuda.empty_cache()
    elif DEV.type == "mps":
        fn = getattr(getattr(torch, "mps", None), "empty_cache", None)
        if fn is not None:
            fn()


ALL_DTYPES = [("fp64", torch.float64), ("fp32", torch.float32),
              ("bf16", torch.bfloat16), ("fp16", torch.float16)]


def torch_version():
    try:
        return tuple(int(x) for x in torch.__version__.split("+")[0].split(".")[:2])
    except Exception:
        return (0, 0)


def candidate_dtypes():
    """Which precisions are worth probing at all on this backend.

    Probing is normally safe -- an unsupported dtype raises and we catch it. MPS
    is the exception: on PyTorch before 2.3 a bfloat16 tensor throws a C++ error
    that kills the interpreter instead of raising, so that combination is filtered
    out here rather than discovered. Apple Silicon has no FP64 at all.
    """
    if DEV.type == "mps":
        keep = [("fp32", torch.float32), ("fp16", torch.float16)]
        if torch_version() >= (2, 3):
            keep.insert(1, ("bf16", torch.bfloat16))
        return keep
    return ALL_DTYPES


def _time_once(fn):
    """Crude single-shot timer: one throwaway call, then one timed call."""
    fn(); sync()
    t0 = time.perf_counter(); fn(); sync()
    return (time.perf_counter() - t0) * 1e3


def gemm_sizes(dtype=torch.float32, start=256, cap_n=16384):
    """Square sizes that fit in memory and stay inside BUDGET_MS."""
    sizes, n = [], start
    while n <= cap_n:
        need = 4 * 3 * n * n * torch.finfo(dtype).bits // 8     # a, b, c + slack
        if need > 0.35 * free_bytes():
            break
        a = torch.randn(n, n, device=DEV, dtype=dtype)
        ms = _time_once(lambda: a @ a)
        del a; free_cache()
        if ms > BUDGET_MS:              # measured, but too slow to sweep with
            if not sizes:
                sizes.append(n)         # ... unless it is all this machine can do
            break
        sizes.append(n)
        n *= 2
    return sizes or [start]


_SIZE_CACHE = {}


def sizes_for(name, dtype):
    """Calibrate once per dtype -- a bf16 CPU GEMM is much slower than fp32."""
    if name not in _SIZE_CACHE:
        _SIZE_CACHE[name] = gemm_sizes(dtype)
    return _SIZE_CACHE[name]


def bw_sizes(candidates=(1, 4, 16, 64, 256, 1024, 4096, 8192)):
    """Buffer sizes in MiB for the bandwidth sweep, capped by free memory."""
    cap = 0.35 * free_bytes()
    return [m for m in candidates if 2 * m * 2**20 < cap] or [1]


print(f"device memory treated as free : {free_bytes()/2**30:.1f} GiB")
print(f"time budget per timed call    : {BUDGET_MS:.0f} ms")
print("precisions worth probing here :", [n for n, _ in candidate_dtypes()])
print("fp32 GEMM sizes for this machine:", sizes_for("fp32", torch.float32))
print("bandwidth buffer sizes (MiB)   :", bw_sizes())


device memory treated as free : 7.3 GiB
time budget per timed call    : 150 ms
precisions worth probing here : ['fp64', 'fp32', 'bf16', 'fp16']
fp32 GEMM sizes for this machine: [256, 512, 1024, 2048, 4096]
bandwidth buffer sizes (MiB)   : [1, 4, 16, 64, 256, 1024]


---
# Part 1 — What am I running on?

## Question 1 (10 pts): Read the device

Print, from Python, every parameter you would need in order to write down the two
roofline numbers — whatever your device happens to be. Then answer, in the markdown cell
below: which of these values is **not** knowable from PyTorch alone, and where would you
get it?

If there is no GPU on this machine, say so and move on: the cell should not pretend to
report compute units it cannot see.

**Names later cells look for:** `INFO` (a dict of what you queried), `CLOCK_GHZ` (a float,
or `None` if nothing reports it) and `DEVICE_NAME` (a string to label your plots).


In [4]:
def device_report(idx=0):
    """Everything PyTorch will tell you about the accelerator it found."""
    if not HAS_GPU:
        return {}
    info = {"backend": ACCEL}
    if IS_CUDA:
        p = torch.cuda.get_device_properties(idx)
        info.update({
            "name":            p.name,
            "arch":            getattr(p, "gcnArchName", None) or f"sm{p.major}{p.minor}",
            "cu_or_sm_count":  p.multi_processor_count,
            "wave_or_warp":    getattr(p, "warp_size", None),
            "total_mem_GiB":   round(p.total_memory / 2**30, 2),
            "vendor_runtime":  f"ROCm {torch.version.hip}" if torch.version.hip
                               else f"CUDA {torch.version.cuda}",
            "n_devices":       torch.cuda.device_count(),
        })
        for extra in ("L2_cache_size", "max_threads_per_multi_processor",
                      "regs_per_multiprocessor", "shared_memory_per_block"):
            if hasattr(p, extra):
                info[extra] = getattr(p, extra)
    else:
        # MPS and the out-of-tree backends expose far less. Take what there is.
        info.update({
            "name": host_name() + f"  (integrated GPU via {ACCEL})",
            "arch": platform.machine(),
            "total_mem_GiB": round(free_bytes() / 2**30, 2),
        })
    return info


def query_clock_ghz():
    """PyTorch does not expose the peak clock. Ask the vendor tool, if it is installed."""
    probes = (
        ["nvidia-smi", "--query-gpu=clocks.max.sm", "--format=csv,noheader,nounits"],
        ["rocm-smi", "--showmaxclocks"],
        ["rocm-smi", "--showclocks"],
    )
    for cmd in probes:
        if shutil.which(cmd[0]) is None:
            continue
        try:
            out = subprocess.run(cmd, capture_output=True, text=True, timeout=30).stdout
        except Exception:
            continue
        mhz = [float(m) for m in re.findall(r"(\d{3,5})\s*[Mm][Hh]z", out)]
        if not mhz:
            mhz = [float(m) for m in re.findall(r"^\s*(\d{3,5})\s*$", out, re.M)]
        if mhz:
            return max(mhz) / 1e3
    return None


INFO = device_report()
if gpu_only():
    print("\nFor the record, the host you are running on:")
    print(f"{'cpu':32s} {host_name()}")
    print(f"{'logical cores':32s} {os.cpu_count()}")
    print(f"{'torch intra-op threads':32s} {torch.get_num_threads()}")
else:
    for k, v in INFO.items():
        print(f"{k:32s} {v}")

CLOCK_GHZ = query_clock_ghz()
DEVICE_NAME = INFO.get("name") or host_name()
print(f"\n{'peak clock (vendor tool)':32s} "
      f"{str(CLOCK_GHZ) + ' GHz' if CLOCK_GHZ else 'not reported -- see Q2'}")
print(f"{'name to label your plots with':32s} {DEVICE_NAME}")

backend                          cuda
name                             NVIDIA GeForce RTX 4060 Laptop GPU
arch                             NVIDIA GeForce RTX 4060 Laptop GPU
cu_or_sm_count                   24
wave_or_warp                     32
total_mem_GiB                    7.62
vendor_runtime                   CUDA 12.6
n_devices                        1
L2_cache_size                    33554432
max_threads_per_multi_processor  1536
regs_per_multiprocessor          65536
shared_memory_per_block          49152

peak clock (vendor tool)         3.105 GHz
name to label your plots with    NVIDIA GeForce RTX 4060 Laptop GPU


## Question 2 (20 pts): Measure your compute ceiling, then read Lecture 4 backwards

Lecture 4 claimed

$$\text{peak FLOP/s} \;=\; \text{CUs} \times \text{clock} \times \text{FLOP per CU per clock}$$

You cannot look up the right-hand side for an arbitrary laptop, and a datasheet peak is
not a number you measured anyway. So run the formula the other way round.

1. **Write the timing harness.** Warm up, synchronise, take a **median** and not a mean,
   and report a spread. Use device events where the backend has them. Everything later in
   the notebook calls this function, so get it right here.
2. **Measure the ceiling.** For every precision your device can actually multiply, time
   the largest square GEMM it will comfortably run and report sustained TFLOP/s. That
   number is *your* compute roof: attainable, on this machine, today.
3. **Invert the formula.** Divide by the CU/SM count you queried in Q1 and by the clock,
   and report the implied FLOP per CU per clock. If either input was unavailable, say so
   and state what you would need to complete the calculation.

Then answer in a markdown cell: **what does the ratio between your FP32 ceiling and your
FP16/BF16 ceiling tell you about the hardware?**

Probe which precisions work. FP64 is missing on Apple Silicon, FP16 matmul
is missing on many CPU builds, and FP8 needs a different API entirely.

**Names later cells look for:** `bench` (your harness, returning median/p5/p95 in ms),
`DTYPES` (the list of usable `(name, dtype)` pairs), `ROOF_TFLOPS` (dtype name -> TFLOP/s)
and `BEST_DT` (the dtype name with the highest ceiling).


In [5]:
# ====================================================================
# YOUR CODE HERE
#
# Implement:
#     bench
#     compute_roof
#
# The specification is in the markdown cell above. Report bandwidth or
# TFLOP/s and the fraction of peak -- not a bare millisecond count.
# ====================================================================

def bench(fn, budget_ms=600, min_rep=3, max_rep=50):
    """Median device time in ms, plus the 5th and 95th percentile.

    The repetition count is chosen from a first timing, so a 20-microsecond
    kernel gets 50 reps and a 200-millisecond laptop GEMM still finishes.
    """
    ms0 = max(_time_once(fn), 1e-4)
    rep = int(min(max_rep, max(min_rep, budget_ms / ms0)))

    for _ in range(min(10, max(2, rep // 5))):        # warm up: clocks, caches, autotune
        fn()
    sync()

    times = []
    if IS_CUDA:
        for _ in range(rep):
            a = torch.cuda.Event(enable_timing=True)
            b = torch.cuda.Event(enable_timing=True)
            a.record(); fn(); b.record()
            torch.cuda.synchronize()
            times.append(a.elapsed_time(b))
    else:
        for _ in range(rep):                          # no events on mps/cpu: sync + clock
            t0 = time.perf_counter(); fn(); sync()
            times.append((time.perf_counter() - t0) * 1e3)

    t = torch.tensor(times, dtype=torch.float64)
    return (t.median().item(), t.quantile(0.05).item(), t.quantile(0.95).item())


def supported_dtypes(candidates=None, n=64):
    """Which precisions this device will actually multiply. Probe, do not assume."""
    ok = []
    for name, dt in (candidates or candidate_dtypes()):
        try:
            x = torch.ones(n, n, device=DEV, dtype=dt)
            y = x @ x
            sync()
            if not torch.isfinite(y.float()).all():
                raise RuntimeError("non-finite result")
            ok.append((name, dt))
            del x, y
        except Exception as e:
            print(f"  {name:5s} unusable here: {type(e).__name__}: {str(e)[:70]}")
    free_cache()
    return ok

def compute_roof(dtypes):
    """Best sustained GEMM throughput per dtype = this device's attainable roof.

    Take the best over every shape that fits, not just the largest one. On a big
    GPU the largest always wins; on a CPU a matrix that still fits in cache can
    beat it, and the ceiling is whichever one actually went fastest. (don't forget to delete tensors and reset the cache on each eval)
    """
    raise NotImplementedError("remove this line and write your solution")


DTYPES = supported_dtypes()
print("usable precisions:", [n for n, _ in DTYPES], "\n")

ROOF = compute_roof(DTYPES)
ROOF_TFLOPS = {k: v["tflops"] for k, v in ROOF.items()}

CU = INFO.get("cu_or_sm_count")
print(f"{'dtype':6s} {'N':>6s} {'ms':>9s} {'+/- ms':>8s} {'TFLOP/s':>10s} "
      f"{'implied FLOP/CU/clk':>21s}")
print("-" * 66)
for name, r in ROOF.items():
    implied = (f"{r['tflops']*1e12/(CU*CLOCK_GHZ*1e9):21.0f}"
               if CU and CLOCK_GHZ else " " * 21)
    print(f"{name:6s} {r['n']:6d} {r['ms']:9.3f} {r['spread']:8.3f} "
          f"{r['tflops']:10.4g} {implied}")

BEST_DT = max(ROOF_TFLOPS, key=ROOF_TFLOPS.get) if ROOF_TFLOPS else None
if BEST_DT:
    print(f"\ncompute roof : {ROOF_TFLOPS[BEST_DT]:.4g} TFLOP/s at {BEST_DT} "
          f"(N={ROOF[BEST_DT]['n']})")
if not (CU and CLOCK_GHZ):
    missing = " and ".join(x for x, ok in
                           (("the CU/SM count", CU), ("the peak clock", CLOCK_GHZ)) if not ok)
    print(f"\nThe last column is blank: {missing} could not be determined here.")
    print("To finish the inversion you would need the vendor product page (clock) and")
    print("the vendor tool or datasheet (core count). Say so in your report -- the")
    print("measured roof above stands on its own, and it is the number the roofline uses.")


usable precisions: ['fp64', 'fp32', 'bf16', 'fp16'] 



NotImplementedError: remove this line and write your solution

## Question 3 (25 pts): Measure achievable memory bandwidth

Using the harness from Q2, measure effective bandwidth with
a large device-to-device copy, and sweep the buffer size.

Report, for each size, the bandwidth in GB/s and its share of the best figure in your
sweep. Then answer: **explain the shape of the curve.** Which point would you put in a
roofline, and why is it not the largest number you measured?

**Names later cells look for:** `BW` (the sweep, a list of dicts) and `ACHIEVED_BW_TBS`
(the figure you would put in a roofline, in TB/s).


In [ ]:
# ====================================================================
#
# Implement:
#     bandwidth_sweep
#
# The specification is in the markdown cell above. Report bandwidth or
# TFLOP/s and the fraction of peak -- not a bare millisecond count.
# ====================================================================
BW_DTYPE = dict(DTYPES).get("fp16") or dict(DTYPES).get("fp32", torch.float32)

def bandwidth_sweep(sizes_mib=None, dtype=None):
    dtype = dtype or BW_DTYPE
    out = []
    for mib in (sizes_mib or bw_sizes()):
        n = int(mib * 2**20 * 8 // torch.finfo(dtype).bits)
        raise NotImplementedError("remove this line and write your solution")
    # Use the bench function to measure the copy speed between a random tensor and an empty desitnation tensor.

BW = bandwidth_sweep()
best = max((r["gbs"] for r in BW), default=float("nan"))

print(f"{'buffer':>10s} {'ms':>9s} {'GB/s':>9s} {'% of best':>10s}")
print("-" * 42)
for r in BW:
    print(f"{r['mib']:7d}MiB {r['ms']:9.3f} {r['gbs']:9.1f} {100*r['gbs']/best:9.1f}%")

ACHIEVED_BW_TBS = (BW[-1]["gbs"] / 1e3) if BW else float("nan")
if BW:
    print(f"\nbest anywhere in the sweep : {best:9.1f} GB/s  at {max(BW, key=lambda r: r['gbs'])['mib']} MiB")
    print(f"largest buffer measured    : {BW[-1]['gbs']:9.1f} GB/s  at {BW[-1]['mib']} MiB"
          "   <-- use THIS in the roofline")
l2 = INFO.get("L2_cache_size")
if l2:
    print(f"\nlast-level cache reported by PyTorch: {l2/2**20:.0f} MiB -- any buffer near")
    print("or below this is partly a cache measurement, not a memory measurement.")

## Question 4 (25 pts): GEMM sweep, and arithmetic intensity

Measure achieved TFLOP/s for square GEMMs across every size and precision the sizing
helper picked for your machine. For each measurement also compute the arithmetic
intensity, so you can place it on a roofline, and the share of the ceiling you measured
in Q2.

Recall from Lecture 4, for an $N \times N \times N$ GEMM:
$\text{FLOPs} = 2N^3$, memory traffic $= 3N^2 \times \text{bytes}$, so
$I = \tfrac{2N}{3\cdot\text{bytes}}$.

**Names later cells look for:** `GEMM`, a list of dicts with at least `dtype`, `n`,
`tflops` and `ai`.


In [ ]:
def gemm_sweep(dtypes=None):
    out = []
    for name, dtype in (dtypes or DTYPES):
        bytes_per_elem = torch.finfo(dtype).bits // 8
        roof = ROOF_TFLOPS.get(name)
        for n in sizes_for(name, dtype):
            a = torch.randn(n, n, device=DEV, dtype=dtype)
            b = torch.randn(n, n, device=DEV, dtype=dtype)
            c = torch.empty(n, n, device=DEV, dtype=dtype)
            ms, lo, hi = bench(lambda a=a, b=b, c=c: torch.mm(a, b, out=c))
            del a, b, c
            free_cache()
            tflops = 2 * n**3 / (ms * 1e9)
            out.append({
                "dtype": name,
                "n": n,
                "ms": ms,
                "spread": (hi - lo) / 2,
                "tflops": tflops,
                "pct": 100 * tflops / roof if roof else float("nan"),
                "ai": (2 * n) / (3 * bytes_per_elem),
            })
    return out

GEMM = gemm_sweep()
print(f"{'dtype':6s} {'N':>6s} {'ms':>9s} {'TFLOP/s':>10s} {'% of roof':>10s} {'I (F/B)':>9s}")
print("-" * 56)
for r in GEMM:
    print(f"{r['dtype']:6s} {r['n']:6d} {r['ms']:9.3f} {r['tflops']:10.4g} "
          f"{r['pct']:9.1f}% {r['ai']:9.0f}")

## Question 5 (20 pts): Build your roofline

Plot the roofline for **your** device from the two numbers you measured: the compute
ceiling from Q2 and the achievable bandwidth from Q3. Put every GEMM measurement from Q4
on it.

Then add, by hand, the arithmetic intensity of two kernels you have not measured: an
element-wise add and a batch-1 decode GEMV. Label which side of the ridge they fall on.

This plot is the deliverable of Lab 0. Labs 1 through 4 all point back to it — including
on a CPU, where the axes read in GFLOP/s but every conclusion is identical.


In [ ]:
def plot_roofline(peak_tflops, bw_tbs, points, title, fname=None):
    fig, ax = plt.subplots(figsize=(8.4, 5.0))
    ai = np.logspace(-1.2, 4.2, 500)
    ax.plot(ai, np.minimum(bw_tbs * ai, peak_tflops), color="#2F5496", lw=2.4,
            label=f"measured roofline: {peak_tflops:.3g} TFLOP/s, {bw_tbs:.3g} TB/s")
    ridge = peak_tflops / bw_tbs
    ax.plot([ridge], [peak_tflops], "o", color="#2F5496", ms=7)
    ax.annotate(f"ridge = {ridge:.0f} FLOP/B", (ridge, peak_tflops),
                xytext=(8, -16), textcoords="offset points", color="#2F5496")

    for p in points:
        ax.plot([p["ai"]], [p["tflops"]], "s", ms=7, color="#B03A22")
        ax.annotate(f"{p['dtype']} N={p['n']}", (p["ai"], p["tflops"]),
                    xytext=(6, 4), textcoords="offset points", fontsize=8.5)

    for label, x in [("element-wise add", 1 / 6), ("decode GEMV, B=1", 1.0)]:
        ax.axvline(x, color="#BBBBBB", ls=":", lw=1.0)
        ax.annotate(label, (x, peak_tflops * 0.5), rotation=90, fontsize=9,
                    ha="right", va="top", color="#444444")

    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel("arithmetic intensity (FLOP/byte)")
    ax.set_ylabel("achieved TFLOP/s")
    ax.set_title(title)
    ax.grid(True, which="major", color="#DDDDDD", lw=0.7)
    ax.legend(loc="lower right", frameon=False, fontsize=9)
    if fname:
        fig.savefig(fname, dpi=160, bbox_inches="tight")
    plt.show()
    return ridge
# Plot the precision with the HIGHEST measured ceiling -- that is the roof this
# machine actually has. On a GPU with matrix cores that is fp16 or bf16; on a CPU
# it is usually fp32, because low precision there is emulated rather than accelerated.
PLOT_DT = BEST_DT or next((d for d in ("fp16", "bf16", "fp32", "fp64")
                           if d in ROOF_TFLOPS), None)
pts = [r for r in GEMM if r["dtype"] == PLOT_DT]

if PLOT_DT and pts and math.isfinite(ACHIEVED_BW_TBS) and ACHIEVED_BW_TBS > 0:
    ROOFLINE_PNG = f"roofline_{PLOT_DT}.png"
    RIDGE = plot_roofline(ROOF_TFLOPS[PLOT_DT], ACHIEVED_BW_TBS, pts,
                          f"{DEVICE_NAME} - measured roofline ({PLOT_DT})", ROOFLINE_PNG)
    print(f"wrote {ROOFLINE_PNG}")
    print(f"compute roof   : {ROOF_TFLOPS[PLOT_DT]:.3g} TFLOP/s ({PLOT_DT})")
    print(f"bandwidth      : {ACHIEVED_BW_TBS*1e3:.1f} GB/s (largest buffer in the sweep)")
    print(f"ridge point    : {RIDGE:.0f} FLOP/byte")
    print(f"\nAnything with arithmetic intensity below {RIDGE:.0f} FLOP/byte cannot reach")
    print("your compute roof, no matter how well it is written.")
else:
    print("Not enough measurements to plot -- rerun Q2 to Q4 first.")


---
# Part 2 — Where does a training step actually go?

Same ResNet and same CIFAR-10 as the Lab 0 you may have seen in previous years. The model
is not the point. The profile is.

Batch size scales with the device so that this part finishes on a laptop. If CIFAR-10
cannot be downloaded, the cell falls back to random tensors of the same shape — the
profile stays valid, only the loss becomes meaningless.


In [ ]:
USE_SYNTHETIC = os.environ.get("LAB0_SYNTHETIC") == "1"
BATCH = 256 if IS_CUDA else (64 if HAS_GPU else 16)
# Worker processes need fork. On macOS and Windows, Python spawns and re-imports
# the main module, which a notebook cell cannot provide -- so load in-process there.
WORKERS = 2 if (platform.system() == "Linux" and (os.cpu_count() or 1) >= 4) else 0


def synthetic_dataset(n=512):
    from torch.utils.data import TensorDataset
    x = torch.randn(n, 3, 32, 32)
    y = torch.randint(0, 10, (n,))
    return {s: TensorDataset(x, y) for s in ("train", "test")}


dataset, SOURCE = None, "CIFAR-10"
if not USE_SYNTHETIC:
    try:
        from torchvision.datasets import CIFAR10
        from torchvision.transforms import (Compose, RandomCrop,
                                            RandomHorizontalFlip, ToTensor)
        transforms = {
            "train": Compose([RandomCrop(32, padding=4), RandomHorizontalFlip(), ToTensor()]),
            "test":  ToTensor(),
        }
        dataset = {s: CIFAR10("data/cifar10", train=(s == "train"), download=True,
                              transform=transforms[s]) for s in ("train", "test")}
    except Exception as e:
        print(f"CIFAR-10 unavailable ({type(e).__name__}: {str(e)[:90]})")
        dataset = None

if dataset is None:
    dataset, SOURCE = synthetic_dataset(), "random tensors (CIFAR-10 not available)"
    print("Falling back to", SOURCE + ".")
    print("The profile below is still valid; the loss value is not.")

loader = {s: DataLoader(dataset[s], batch_size=BATCH, shuffle=(s == "train"),
                        num_workers=WORKERS, pin_memory=IS_CUDA,
                        drop_last=(s == "train"))
          for s in ("train", "test")}

print(f"data source : {SOURCE}")
print(f"batch size  : {BATCH}    dataloader workers: {WORKERS}")
print({s: len(loader[s]) for s in loader}, "batches")


In [ ]:
class RESNET(nn.Module):

    def __init__(self):
        super(RESNET, self).__init__()
        in_channel = 3
        num_classes = 10
        self.channel_1 = 64
        self.channel_2 = 128
        self.channel_3 = 256
        self.channel_4 = 512

        self.conv_1 = nn.Conv2d(in_channel, self.channel_1, (7, 7), padding=3, stride=2)
        self.bn1 = nn.BatchNorm2d(self.channel_1)
        self.block_1 = nn.Sequential(
            nn.Conv2d(self.channel_1, self.channel_1, (3, 3), padding=1, stride=1),
            nn.BatchNorm2d(self.channel_1),
            nn.ReLU(),
            nn.Conv2d(self.channel_1, self.channel_1, (3, 3), padding=1, stride=1),
        )

        self.bn2 = nn.BatchNorm2d(self.channel_1)
        self.block_2 = nn.Sequential(
            nn.Conv2d(self.channel_1, self.channel_2, (3, 3), padding=1, stride=1),
            nn.BatchNorm2d(self.channel_2),
            nn.ReLU(),
            nn.Conv2d(self.channel_2, self.channel_2, (3, 3), padding=1, stride=1),
        )

        self.bn3 = nn.BatchNorm2d(self.channel_2)
        self.conv_2_skip = nn.Conv2d(self.channel_1, self.channel_2, (1, 1), stride=1)
        self.block_3 = nn.Sequential(
            nn.Conv2d(self.channel_2, self.channel_3, (3, 3), padding=1, stride=1),
            nn.BatchNorm2d(self.channel_3),
            nn.ReLU(),
            nn.Conv2d(self.channel_3, self.channel_3, (3, 3), padding=1, stride=1),
        )

        self.bn4 = nn.BatchNorm2d(self.channel_3)
        self.conv_3_skip = nn.Conv2d(self.channel_2, self.channel_3, (1, 1), stride=1)
        self.block_4 = nn.Sequential(
            nn.Conv2d(self.channel_3, self.channel_4, (3, 3), padding=1, stride=2),
            nn.BatchNorm2d(self.channel_4),
            nn.ReLU(),
            nn.Conv2d(self.channel_4, self.channel_4, (3, 3), padding=1, stride=1),
        )

        self.bn5 = nn.BatchNorm2d(self.channel_4)
        self.conv_4_skip = nn.Conv2d(self.channel_3, self.channel_4, (1, 1), stride=2)

        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d((3, 3), padding=1, stride=2)
        self.avgpool = nn.AdaptiveAvgPool2d(output_size=(1, 1))
        self.fc = nn.Linear(self.channel_4, num_classes)

    def forward(self, x):
        x = self.conv_1(x)
        x = self.bn1(x)
        x = self.relu(x)
        maxpool_out = self.maxpool(x)
        x = self.block_1(maxpool_out)
        x = self.bn2(x)
        block1_out = x + maxpool_out
        x = self.block_2(block1_out)
        x = self.bn3(x)
        block2_out = x + self.conv_2_skip(block1_out)
        x = self.block_3(block2_out)
        x = self.bn4(x)
        block3_out = x + self.conv_3_skip(block2_out)
        x = self.block_4(block3_out)
        x = self.bn5(x)
        block4_out = x + self.conv_4_skip(block3_out)
        x = self.avgpool(block4_out)
        x = x.view(x.shape[0], -1)
        scores = self.fc(x)
        return scores

In [ ]:
model = RESNET().to(DEV)
criterion = nn.CrossEntropyLoss()
optimizer = SGD(model.parameters(), lr=0.4, momentum=0.9, weight_decay=5e-4)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"trainable parameters : {n_params/1e6:.2f} M")
print(f"weights at FP32      : {4*n_params/2**20:.1f} MiB")

# Rough per-step cost, so we know what to expect before we measure it.
# One forward+backward is ~3x the forward FLOPs.
try:
    from torchprofile import profile_macs
    macs = profile_macs(model, torch.zeros(1, 3, 32, 32, device=DEV))
    print(f"MACs per image       : {macs/1e6:.1f} M")
    print(f"est. FLOPs per step  : {3*2*macs*BATCH/1e9:.1f} GFLOP")
except Exception as e:
    print("torchprofile unavailable:", type(e).__name__)
    macs = None

# Compare against the ceiling you measured in Q2 -- if you kept those names.
if macs and "ROOF_TFLOPS" in globals() and globals().get("BEST_DT"):
    est_ms = 3 * 2 * macs * BATCH / (ROOF_TFLOPS[BEST_DT] * 1e12) * 1e3
    print(f"at your {BEST_DT} roof that step would take {est_ms:.1f} ms if it ran at peak;")
    print("Q6 measures what it actually takes, and the gap is the subject of the course.")


## Question 6 (20 pts): Profile one step and categorise the time

Profile a few training steps with `torch.profiler`. Then report the share of time spent in
each of: GEMM/convolution, element-wise and normalisation, reductions, data movement, the
optimizer, and everything else.

State which single kernel category you would attack first, and why.

On a machine with no profiler support for the device, `torch.profiler` still reports
host-side operator time. The categories mean the same thing; say which one you measured.

**Names the final cell looks for:** `tot` (category -> microseconds), `grand` (the total,
in microseconds) and `USED_DEVICE_TIME` (a bool: device-side or host-side time).


In [ ]:
from torch.profiler import profile, ProfilerActivity, schedule


def cycle(dl):
    """Endless batches, so a benchmark loop never trips over StopIteration."""
    while True:
        for batch in dl:
            yield batch


def train_step(it):
    x, y = next(it)
    x, y = x.to(DEV, non_blocking=True), y.to(DEV, non_blocking=True)
    optimizer.zero_grad(set_to_none=True)
    loss = criterion(model(x), y)
    loss.backward()
    optimizer.step()
    return loss


it = cycle(loader["train"])
model.train()

try:
    acts = list(torch.profiler.supported_activities())
except Exception:
    acts = [ProfilerActivity.CPU]
print("profiler activities:", [str(a).split(".")[-1] for a in acts])

ACTIVE = 5
with profile(activities=acts,
             schedule=schedule(wait=1, warmup=3, active=ACTIVE, repeat=1),
             record_shapes=True) as prof:
    for _ in range(1 + 3 + ACTIVE):
        train_step(it)
        prof.step()

# Attribute names moved between PyTorch versions; try the current one first.
keys = (("self_device_time_total", "self_cuda_time_total", "self_cpu_time_total")
        if HAS_GPU else ("self_cpu_time_total",))
for sort_key in keys:
    try:
        print(prof.key_averages().table(sort_by=sort_key, row_limit=18))
        break
    except Exception:
        continue
prof.export_chrome_trace("lab0_trace.json")
print("\nwrote lab0_trace.json  (open in Perfetto, or in AMD TraceLens)")


In [ ]:
# Keywords cover ROCm kernel names, CUDA kernel names, and the aten:: operator
# names you get on a CPU-only profile. Extend the lists until "other" is small.
CATEGORIES = {
    "GEMM / conv":        ("gemm", "conv", "implicit", "cutlass", "wgrad", "dgrad",
                           "Cijk", "sgemm", "ampere", "gemv", "mfma", "addmm",
                           "::mm", "bmm", "linear", "mkldnn"),
    "elementwise / norm": ("elementwise", "batch_norm", "bn_", "relu", "add", "mul",
                           "vectorized", "unrolled_elementwise", "native_layer_norm",
                           "threshold", "clamp", "fill", "zero_"),
    "reduction":          ("reduce", "sum", "softmax", "nll", "cross_entropy", "max_pool",
                           "adaptive_avg", "mean"),
    "data movement":      ("copy", "memcpy", "cat", "contiguous", "permute", "transpose",
                           "to_", "item"),
    "optimizer":          ("sgd", "adam", "foreach", "multi_tensor"),
}


def _self_time(e, use_device):
    if use_device:
        for attr in ("self_device_time_total", "self_cuda_time_total"):
            t = getattr(e, attr, 0) or 0
            if t:
                return t
        return 0.0
    return e.self_cpu_time_total


def categorise(prof):
    evs = list(prof.key_averages())
    use_device = HAS_GPU and sum(_self_time(e, True) for e in evs) > 0
    if not use_device:
        print("No device-side timing available (CPU run, or this backend has no profiler")
        print("support), so the shares below are host-side operator time. The categories")
        print("mean the same thing; say which one you measured.\n")

    tot = {k: 0.0 for k in CATEGORIES}
    tot["other"] = 0.0
    grand = 0.0
    for e in evs:
        t = _self_time(e, use_device)
        if t <= 0:
            continue
        grand += t
        low = e.key.lower()
        for cat, keys in CATEGORIES.items():
            if any(k.lower() in low for k in keys):
                tot[cat] += t
                break
        else:
            tot["other"] += t
    return tot, grand, use_device


tot, grand, USED_DEVICE_TIME = categorise(prof)
print(f"{'category':22s} {'ms':>10s} {'share':>8s}   "
      f"({'device' if USED_DEVICE_TIME else 'host'} self time)")
print("-" * 54)
for k, v in sorted(tot.items(), key=lambda kv: -kv[1]):
    if grand:
        print(f"{k:22s} {v/1e3:10.2f} {100*v/grand:7.1f}%")
print("-" * 54)
print(f"{'TOTAL':22s} {grand/1e3:10.2f}")


### Question 6, second half: kernel-bound or gap-bound?

The categories above only account for time that was *spent in an operator*. Compare their
total against the wall-clock time of one step, measured with your Q2 harness, and report
the unaccounted fraction. Then say what that fraction implies about where the next
optimisation should go.


In [ ]:
# ====================================================================
# YOUR CODE HERE
#
# The specification is in the markdown cell above. Report bandwidth or
# TFLOP/s and the fraction of peak -- not a bare millisecond count.
# ====================================================================
 # ms_step, lo, hi = bench ...
raise NotImplementedError("remove this line and write your solution")
step_ms = grand / 1e3 / ACTIVE          # profiled self time per active step
print(f"wall clock per step : {ms_step:8.2f} ms")
print(f"profiled self time  : {step_ms:8.2f} ms "
      f"({'device' if USED_DEVICE_TIME else 'host'})")
print(f"unaccounted         : {ms_step - step_ms:8.2f} ms "
      f"({100*(ms_step-step_ms)/ms_step:.0f}%)")
print("\nIf 'unaccounted' is large, the fix is host-side (dataloader workers, pinned")
print("memory, launch overhead, CUDA/HIP graphs) -- not a faster kernel.")
print("On a CPU-only profile the two figures can also exceed each other, because host")
print("operator time nests; say so rather than reporting a negative gap as a result.")

# Note that this lab is for your own reference and it serves as an introduction to the course. 

### You are not required to upload this lab.